# DNA Methylation Analysis with `cytozip`

CLI walkthrough using 9 mouse 3C-seq cells shipped under
`cytozip_example_data/bam/`. All steps are demonstrated as shell
commands (`! czip ...`); benchmark results produced offline by the
scripts under `tests/` are loaded and rendered as tables.

1. Setup + reference `.cz`
2. BAM → `.cz`  (`czip bam_to_cz`)
3. ALLC → `.cz` (`czip allc2cz`) + benchmark table (BAM→ALLC vs BAM→CZ, ALLC→CZ)
4. `view` / `query` (local) + benchmark table (tabix vs czip query)
5. `view` / `query` a remote `.cz`
6. `catcz` + `merge_cz`
7. `cz_to_anndata`

Benchmark
```python
python tests/benchmark_bam_to_cz.py  -j 20
python tests/benchmark_allc_to_cz.py -j 20
python tests/benchmark_query.py
```

In [1]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))

## 1. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.


In [3]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-j JOBS]
                      [--keep_temp] [--no_delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -j JOBS, --jobs JOBS  number of parallel processes (CPUs) (default: 12)
  --keep_temp           keep temp directory (default: False)
  --no_delta            disable DELTA encoding on the pos column (default: on,
                        gives ~3x smaller reference files with mild query
                        overhead) (default: False)


In [4]:
!time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/mm10_with_chrL.allc.cz -j 20

2026-04-25 20:51:35.755 | DEBUG    | cytozip.allc:WriteC:62 - chr1
2026-04-25 20:51:36.455 | DEBUG    | cytozip.allc:WriteC:62 - chr10
2026-04-25 20:51:37.129 | DEBUG    | cytozip.allc:WriteC:62 - chr11
2026-04-25 20:51:37.789 | DEBUG    | cytozip.allc:WriteC:62 - chr12
2026-04-25 20:51:38.431 | DEBUG    | cytozip.allc:WriteC:62 - chr13
2026-04-25 20:51:39.104 | DEBUG    | cytozip.allc:WriteC:62 - chr14
2026-04-25 20:51:39.646 | DEBUG    | cytozip.allc:WriteC:62 - chr15
2026-04-25 20:51:40.170 | DEBUG    | cytozip.allc:WriteC:62 - chr16
2026-04-25 20:51:40.685 | DEBUG    | cytozip.allc:WriteC:62 - chr17
2026-04-25 20:51:41.159 | DEBUG    | cytozip.allc:WriteC:62 - chr18
2026-04-25 20:51:41.464 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456210_random
2026-04-25 20:51:41.466 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456211_random
2026-04-25 20:51:41.466 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456212_random
2026-04-25 20:51:41.467 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456

In [5]:
!czip header -I output/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  1316257876
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [6]:
! czip view -I output/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


In [7]:
! czip summary -I output/mm10_with_chrL.allc.cz | head

chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	102	94758324	94772911	904	78962721
chr10	94772911	62930025	157712606	603	52609184
chr11	157712606	60400150	218122314	596	52027265
chr12	218122314	58107969	276239249	559	48799752
chr13	276239249	58271673	334519872	558	48750883
chr14	334519872	59989460	394518522	573	49987736
chr15	394518522	50122390	444648678	484	42230765
chr16	444648678	46787668	491443504	446	38899643
chr17	491443504	45991169	537441879	449	39153472


## 3. BAM → `.cz`

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


In [8]:
!czip bam_to_cz --help

usage: czip bam_to_cz [-h] -I INPUT -g GENOME [-O OUTPUT]
                      [--num_upstr_bases NUM_UPSTR_BASES]
                      [--num_downstr_bases NUM_DOWNSTR_BASES]
                      [--min_mapq MIN_MAPQ]
                      [--min_base_quality MIN_BASE_QUALITY] [-c BATCH_SIZE]
                      [--convert_bam_strandness] [--save_count_df]
                      [--mode {full,pos_mc_cov,mc_cov}]
                      [--count_fmt {B,H,I,Q}] [-r REFERENCE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input position-sorted BAM (bismark/hisat-3n) (default:
                        None)
  -g GENOME, --genome GENOME
                        indexed reference fasta (.fai required) (default:
                        None)
  -O OUTPUT, --output OUTPUT
                        output .cz path (default: <bam_stem>.cz) (default:
                        None)
  --num_upstr_bases NUM_UPSTR_BASES
              

In [9]:
! time czip bam_to_cz -I bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam \
 --genome ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/cz/FC_M_P12b_3C_2-5-M17-N10.cz \
 --mode mc_cov --count_fmt B --reference output/mm10_with_chrL.allc.cz

/home/x-wding2/Software/conda/m3c/lib/python3.10/site-packages/cytozip/__init__.py:702: UserWarning: mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
  bam_to_cz(bam_path=args.input, genome=args.genome,

real	3m6.573s
user	3m41.758s
sys	0m10.720s


bam to allc
```python
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam", reference_fasta=os.path.expanduser("~/Ref/mm10/mm10_ucsc_with_chrL.fa"), output_path="output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz", cpu=1,num_upstr_bases=0, num_downstr_bases=2, min_mapq=10, min_base_quality=20, compress_level=5)
```

### BAM → `.cz` benchmark vs ALLCools

Produced by `tests/benchmark_bam_to_cz.py -j 9`. ALLCools writes a
gzipped TSV (`.allc.tsv.gz`); cytozip writes a chunked, columnar
zstd-compressed `.cz`.


In [10]:
import pandas as pd
bam_bench = pd.read_csv('output/bam_benchmark/bam_benchmark.tsv', sep='\t')
bam_bench.round(2)

,cell,bam_size_mb,n_reads,allc_wall_s,allc_rss_mb,allc_size_mb,cz_wall_s,cz_rss_mb,cz_size_mb,speedup,size_ratio
0,FC_E17b_3C_5-5-I24-A21,685.99,7330382,1709.14,571.62,442.62,504.06,919.16,78.67,3.39,0.18
1,FC_M_E15a_3C_1-1-I5-B1,177.37,1787812,707.73,571.79,102.53,543.51,917.24,22.24,1.30,0.22
2,FC_M_P12b_3C_2-5-M17-N10,239.89,2307326,930.83,573.43,143.12,584.09,919.04,29.69,1.59,0.21
3,FC_M_P3b_3C_6-6-J3-P24,147.36,1395949,651.52,573.56,85.71,507.20,917.20,19.34,1.28,0.23
4,FC_M_P6a_3C_7-3-K21-P5,89.76,938870,438.92,574.16,48.84,404.69,919.00,12.16,1.08,0.25
5,FC_M_P9B_3C_6-2-F6-O4,32.28,342778,249.22,572.84,17.61,298.59,923.62,6.02,0.83,0.34
6,FC_P0b_3C_5-1-I24-J14,485.51,5028030,1486.54,572.81,285.50,428.59,917.18,52.55,3.47,0.18
7,FC_P13a_3C_7-1-A11-O1,260.71,2588754,1006.73,573.65,155.84,552.49,920.95,31.30,1.82,0.20
8,FC_P28a_3C_2-1-E5-N14,186.32,1908846,820.91,574.05,112.95,551.42,917.14,23.94,1.49,0.21


In [11]:
!cat output/bam_benchmark/bam_benchmark.txt

cytozip bam_to_cz  vs  ALLCools bam_to_allc
reference FASTA : /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
reference .cz   : /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/mm10_with_chrL.allc.cz
BAMs            : 9   total reads = 23,628,747

ALLCools : time=  8001.5 s   size=  1394.7 MB
cytozip  : time=  4374.6 s   size=   275.9 MB

speedup (allc / cz time)  =  1.83x
compression (cz / allc)   =  19.8%


## 4. ALLC → `.cz`

Pack an existing `.allc.tsv.gz` into `.cz`. Two layouts:

In [12]:
! czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP]
                    [--chrom_order CHROM_ORDER] [-c BATCH_SIZE]
                    [--sort_col SORT_COL] [--delta_cols DELTA_COLS] [-j JOBS]
                    [--pattern PATTERN] [--no_skip_existing]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz, OR a directory containing many
                        allc.tsv.gz (batch mode: --output must be a directory)
                        (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (single-file), or output directory
                        (batch mode) (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (

In [13]:
# Standalone (positions inside the .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz \
      -F Q,B,B -C pos,mc,cov -u 1,4,5

2026-04-25 20:55:18.112 | INFO     | cytozip.allc:allc2cz:263 - /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz

real	0m29.882s
user	0m24.110s
sys	0m2.428s


In [14]:
# Reference-aligned (positions come from the reference .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
      -r output/mm10_with_chrL.allc.cz

2026-04-25 20:55:48.136 | INFO     | cytozip.allc:allc2cz:263 - /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz

real	1m6.757s
user	0m56.338s
sys	0m10.149s


In [15]:
! ls output/allc/FC_M_P12b_3C_2-5-M17-N10* -sh

137M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz
1.5M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz.tbi
 32M output/allc/FC_M_P12b_3C_2-5-M17-N10.cz
 42M output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz


## 5. `view` and `query`

`czip view` streams an entire dimension (e.g. one chrom). `czip query`
pulls a region. Both can use `-r REFERENCE` to expand reference-aligned
files into full ALLC-style records (chrom / pos / context / mc / cov).


In [16]:
! czip view -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz --show_dims 0 | awk '$6>0' | head

chrom	pos	strand	context	mc	cov
chr1	3005823	+	CTT	0	1
chr1	3005826	+	CTA	0	1
chr1	3005836	+	CAA	0	1
chr1	3005840	+	CCA	0	1
chr1	3005841	+	CAC	0	1
chr1	3005843	+	CCA	0	1
chr1	3005844	+	CAC	0	1
chr1	3005846	+	CAA	1	1
chr1	3005850	+	CCA	0	1


In [17]:
! tabix output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz chr9:3000294-3005294 | head

chr9	3000294	-	CAT	15	23	1
chr9	3000296	+	CCT	51	62	1
chr9	3000297	+	CTA	53	63	1
chr9	3000300	+	CAA	52	65	1
chr9	3000304	-	CAT	2	26	1
chr9	3000305	-	CCA	1	27	1
chr9	3000307	+	CAT	60	71	1
chr9	3000312	+	CTA	63	76	1
chr9	3000321	+	CCA	67	78	1
chr9	3000322	+	CAA	65	76	1
Failed to write to stdout: Broken pipe


In [18]:
! time czip query -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 -s 3000294 -e 3005294 | head

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65
chr9	3000304	-	CAT	2	26
chr9	3000305	-	CCA	1	27
chr9	3000307	+	CAT	60	71
chr9	3000312	+	CTA	63	76
chr9	3000321	+	CCA	67	78

real	0m0.253s
user	0m0.143s
sys	0m0.093s


## 6. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [32]:
! czip header -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  29686076
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [33]:
# view a cz file (no coordinates were stored) alone
! czip view -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [34]:
# query remote .cz file with local reference .cz:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 \
    -s 3000294 -e 3005294 | head -n 5

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m1.067s
user	0m0.217s
sys	0m0.114s


In [35]:
# or both the cz and the reference can be accessed via HTTP(S) URLs, without downloading any files locally:
! time czip query -I https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r https://neomorph.salk.edu/ftp/bican/mm10_with_chrL.allc.cz \
    -K chr9 -s 3000294 -e 3005294 | head -n 5
# query 5000 bp of regions containing 1759 records

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65

real	0m2.137s
user	0m0.240s
sys	0m0.135s


## 7. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [38]:
! ls output/cz/*.cz -sh

 76M output/cz/FC_E17b_3C_5-5-I24-A21.cz
 22M output/cz/FC_M_E15a_3C_1-1-I5-B1.cz
 29M output/cz/FC_M_P12b_3C_2-5-M17-N10.cz
 19M output/cz/FC_M_P3b_3C_6-6-J3-P24.cz
 12M output/cz/FC_M_P6a_3C_7-3-K21-P5.cz
6.0M output/cz/FC_M_P9B_3C_6-2-F6-O4.cz
 51M output/cz/FC_P0b_3C_5-1-I24-J14.cz
 30M output/cz/FC_P13a_3C_7-1-A11-O1.cz
 23M output/cz/FC_P28a_3C_2-1-E5-N14.cz


In [39]:
! time czip catcz -O output/all_cells.cz -I "output/cz/*.cz" --key_added cell_id -F B,B -C mc,cov


real	0m0.454s
user	0m0.204s
sys	0m0.214s


In [40]:
! czip header -I output/all_cells.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  275929173
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom', 'cell_id']
header_size  :  47


In [41]:
! czip summary -I output/all_cells.cz | head

chrom	cell_id	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	FC_E17b_3C_5-5-I24-A21	47	5989004	5990303	151	78962721
chr10	FC_E17b_3C_5-5-I24-A21	5990303	3991267	9982423	101	52609184
chr11	FC_E17b_3C_5-5-I24-A21	9982423	3827680	13810948	100	52027265
chr12	FC_E17b_3C_5-5-I24-A21	13810948	3610272	17422017	94	48799752
chr13	FC_E17b_3C_5-5-I24-A21	17422017	3696490	21119296	93	48750883
chr14	FC_E17b_3C_5-5-I24-A21	21119296	3669883	24789992	96	49987736
chr15	FC_E17b_3C_5-5-I24-A21	24789992	3165986	27956671	81	42230765
chr16	FC_E17b_3C_5-5-I24-A21	27956671	2933049	30890365	75	38899643
chr17	FC_E17b_3C_5-5-I24-A21	30890365	2871092	33762102	75	39153472


In [27]:
# merge 9 single-cell .cz files into a pseudobulk .cz file, summing the mc and cov values across all cells:
! time czip merge_cz -i output/cz -O output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -F H,H -j 20

2026-04-25 20:57:18.907 | INFO     | cytozip.merge:merge_cz:492 - output/pseudobulk.cz
2026-04-25 20:57:28.203 | INFO     | cytozip.merge:_bg_rmtree:141 - Removing temp dir /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz.tmp (in background)

real	0m9.509s
user	1m37.793s
sys	0m13.934s


In [28]:
! czip view -I output/pseudobulk.cz --show_dims 0 \
    -r output/mm10_with_chrL.allc.cz | awk '$6 >10' | head

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520474	-	CAG	13	13
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [29]:
! czip query -I output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520459	+	CCC	2	10
chr1	25520460	+	CCT	2	10
chr1	25520461	+	CTG	0	10
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520465	+	CGT	7	9
chr1	25520466	-	CGC	7	7
chr1	25520469	+	CCC	1	10
chr1	25520470	+	CCC	0	10
chr1	25520471	+	CCT	0	10
chr1	25520472	+	CTG	1	10
chr1	25520474	-	CAG	13	13
chr1	25520477	+	CCG	0	9
chr1	25520478	+	CGG	5	10
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [30]:
! for file in `ls output/cz`; do echo ${file} && czip query -I output/cz/${file} -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482; done;

FC_E17b_3C_5-5-I24-A21.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	3	3
chr1	25520458	-	CCA	3	3
chr1	25520459	+	CCC	2	2
chr1	25520460	+	CCT	2	2
chr1	25520461	+	CTG	0	2
chr1	25520463	-	CAG	3	3
chr1	25520464	-	CCA	3	3
chr1	25520465	+	CGT	1	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	1	2
chr1	25520470	+	CCC	0	2
chr1	25520471	+	CCT	0	2
chr1	25520472	+	CTG	1	2
chr1	25520474	-	CAG	3	3
chr1	25520477	+	CCG	0	2
chr1	25520478	+	CGG	1	2
chr1	25520479	-	CGG	3	3
chr1	25520480	-	CCG	3	3
chr1	25520481	-	CCC	3	3
chr1	25520482	-	CCC	3	3
FC_M_E15a_3C_1-1-I5-B1.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	1	1
chr1	25520458	-	CCA	1	1
chr1	25520459	+	CCC	0	1
chr1	25520460	+	CCT	0	1
chr1	25520461	+	CTG	0	1
chr1	25520463	-	CAG	1	1
chr1	25520464	-	CCA	1	1
chr1	25520465	+	CGT	0	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	0	1
chr1	25520470	+	CCC	0	1
chr1	25520471	+	CCT	0	1
chr1	25520472	+	CTG	0	1
chr1	25520474	-	CAG	2	2
chr1	25520477	+	CCG	0	0
chr1	25520478	+	CGG	0	1
chr1	25520479	-	CGG	

## 8. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [42]:
! time czip cz_to_anndata -I output/cz \
    -f /home/x-wding2/Ref/mm10/annotations/gencode.vM23.annotation.gtf \
    -O output/allcells.h5ad -r output/mm10_with_chrL.allc.cz -j 10


real	0m44.854s
user	2m45.786s
sys	1m46.240s


In [43]:
import anndata
adata=anndata.read_h5ad("output/allcells.h5ad")
adata

AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'cov', 'mc'

In [44]:
adata.obs

,alpha,beta,prior_mean
FC_E17b_3C_5-5-I24-A21,16.560707,4.300688,0.793845
FC_M_E15a_3C_1-1-I5-B1,2.186391,2.139432,0.505428
FC_M_P12b_3C_2-5-M17-N10,2.996044,2.876152,0.510208
FC_M_P3b_3C_6-6-J3-P24,1.944797,1.912777,0.504150
FC_M_P6a_3C_7-3-K21-P5,1.220374,1.223111,0.499440
FC_M_P9B_3C_6-2-F6-O4,0.856140,0.833441,0.506717
FC_P0b_3C_5-1-I24-J14,5.304654,5.270510,0.501614
FC_P13a_3C_7-1-A11-O1,2.719304,2.712830,0.500596
FC_P28a_3C_2-1-E5-N14,2.281293,2.224639,0.506287


In [45]:
adata.var

,chrom,start,end,gene_id,gene_name,gene_type,strand
name,,,,,,,
4933401J01Rik,chr1,3071252,3076322,ENSMUSG00000102693.1,4933401J01Rik,TEC,+
Gm26206,chr1,3100015,3104125,ENSMUSG00000064842.1,Gm26206,snRNA,+
Xkr4,chr1,3203900,3673498,ENSMUSG00000051951.5,Xkr4,protein_coding,-
Gm18956,chr1,3250756,3255236,ENSMUSG00000102851.1,Gm18956,processed_pseudogene,+
Gm37180,chr1,3363730,3370549,ENSMUSG00000103377.1,Gm37180,TEC,-
...,...,...,...,...,...,...,...
mt-Nd6,chrM,11551,16070,ENSMUSG00000064368.1,mt-Nd6,protein_coding,-
mt-Te,chrM,12070,16139,ENSMUSG00000064369.1,mt-Te,Mt_tRNA,-
mt-Cytb,chrM,12144,17288,ENSMUSG00000064370.1,mt-Cytb,protein_coding,+


In [46]:
adata.to_df()

name,4933401J01Rik,Gm26206,Xkr4,Gm18956,Gm37180,Gm37363,Gm37686,Gm1992,Gm37329,Gm7341,...,mt-Nd4,mt-Th,mt-Ts2,mt-Tl2,mt-Nd5,mt-Nd6,mt-Te,mt-Cytb,mt-Tt,mt-Tp
FC_E17b_3C_5-5-I24-A21,0.614130,0.750000,0.785559,0.774510,0.793548,0.779528,0.666667,0.741276,0.552301,0.751807,...,0.896099,0.891881,0.894037,0.897048,0.894648,0.890524,0.892887,0.887724,0.881818,0.884255
FC_M_E15a_3C_1-1-I5-B1,0.142857,0.606742,0.446833,0.000000,0.000000,0.306452,0.188679,0.325444,0.696429,0.386076,...,0.076923,0.076923,0.428571,0.647059,0.600000,0.600000,0.600000,0.600000,0.719298,0.719298
FC_M_P12b_3C_2-5-M17-N10,0.000000,0.565789,0.499050,0.640000,0.460938,0.456522,0.513889,0.451348,0.500000,0.421875,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.157895,1.000000,1.000000
FC_M_P3b_3C_6-6-J3-P24,0.812500,0.438356,0.489824,0.000000,0.661765,0.468085,0.390476,0.454992,0.509091,0.163043,...,0.687500,0.687500,0.687500,0.687500,0.687500,0.666667,0.666667,0.666667,0.000000,0.000000
FC_M_P6a_3C_7-3-K21-P5,0.083333,0.000000,0.538052,0.000000,0.617647,0.000000,1.000000,0.628931,1.000000,0.578125,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
FC_M_P9B_3C_6-2-F6-O4,0.000000,0.000000,0.541885,0.000000,0.459016,0.000000,1.000000,0.565789,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
FC_P0b_3C_5-1-I24-J14,0.452632,0.739130,0.489069,0.227273,0.380403,0.504762,0.308943,0.497350,0.423645,0.510309,...,0.476415,0.459075,0.476190,0.508050,0.577723,0.572570,0.591398,0.598462,0.686076,0.687817
FC_P13a_3C_7-1-A11-O1,0.453901,0.384615,0.523233,0.348624,0.556604,0.419355,0.569378,0.542421,0.700000,0.380952,...,0.567989,0.545879,0.545879,0.539171,0.492574,0.473748,0.437414,0.442623,0.322581,0.327869
FC_P28a_3C_2-1-E5-N14,0.698630,0.000000,0.492707,0.550388,0.417476,0.071429,0.615385,0.514677,0.402439,0.058824,...,0.456328,0.430020,0.435318,0.439834,0.461176,0.512586,0.513707,0.516588,0.535836,0.535836
